# Validation Sample


In [ ]:
# === notebook config (auto-managed; edit values, not the tag) ===
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Database
DB_PATH = "../data/humans_clean.sqlite3"

# Figure style — minimal, Nature/Science publication standard
FIGSIZE = (8, 5)
DPI = 120
FONT_TITLE = 16
FONT_LABEL = 13
FONT_TICK = 11
FONT_LEGEND = 10

# Light, restrained palette (avoid AI-slop saturation)
COLOR_PRIMARY = "#2171b5"
COLOR_SECONDARY = "#b5542a"
COLOR_NEUTRAL = "#7f7f7f"
COLOR_LIGHT = "#d9d9d9"
COLOR_ACCENT = "#6a9e3a"
PALETTE = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_ACCENT, COLOR_NEUTRAL, COLOR_LIGHT]

import matplotlib as _mpl
_mpl.rcParams.update({
    "figure.figsize": FIGSIZE,
    "figure.dpi": DPI,
    "axes.titlesize": FONT_TITLE,
    "axes.labelsize": FONT_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})


## Validation Sample: Stratified Selection of 500 Individuals

This notebook generates a stratified random sample of 500 individuals for manual validation of the Cliopatria linkage.
The sample is stratified along five dimensions:

- **Method**: polygon, url, url_fallback
- **Origin**: nationality, birthplace, deathplace
- **Region**: macro-regions (Western Europe, Asia, etc.)
- **Era**: Ancient, Medieval, Early Modern, Modern, Contemporary
- **Notability**: Low / Medium / High (based on external identifier count)

### Define sampling targets and stratification bins

Quota targets per (method, origin) stratum are set to reach 500 total individuals,
with heavier weight on polygon-nationality (the most common linkage method).

In [ ]:
import sqlite3
import csv
import os
from collections import defaultdict, Counter

random.seed(42)

DB = DB_PATH

# Sampling targets per (method, origin)
TARGETS = {
    ('polygon', 'nationality'): 120,
    ('polygon', 'birthplace'): 60,
    ('polygon', 'deathplace'): 40,
    ('url', 'nationality'): 30,
    ('url', 'birthplace'): 20,
    ('url', 'deathplace'): 15,
    ('url_fallback', 'nationality'): 120,
    ('url_fallback', 'birthplace'): 55,
    ('url_fallback', 'deathplace'): 40,
}

TIME_ERAS = [
    ('Ancient', -5000, 499),
    ('Medieval', 500, 1399),
    ('Early Modern', 1400, 1699),
    ('Modern', 1700, 1899),
    ('Contemporary', 1900, 2100),
]

NOTABILITY_BINS = [
    ('Low', 0, 2),
    ('Medium', 3, 10),
    ('High', 11, 9999),
]

def get_era(year):
    if year is None:
        return 'Unknown'
    for name, lo, hi in TIME_ERAS:
        if lo <= year <= hi:
            return name
    return 'Unknown'

def get_notability(count):
    if count is None:
        return 'Unknown'
    for name, lo, hi in NOTABILITY_BINS:
        if lo <= count <= hi:
            return name
    return 'Unknown'

print(f"Target: {sum(TARGETS.values())} individuals")
for k, v in TARGETS.items():
    print(f"  {k[0]:15s} - {k[1]:12s}: {v}")


### Draw stratified samples from the database

For each (method, origin) stratum, individuals are grouped into sub-strata by region, era, and notability.
An equal number of individuals is drawn from each sub-stratum; any remaining quota is filled from the leftover pool.

In [2]:
conn = sqlite3.connect(DB)
conn.execute('PRAGMA cache_size=-500000')

# Pre-load lookups
polity_urls = {}
for pid, url in conn.execute('SELECT id, wikipedia_url FROM polities_cliopatria'):
    polity_urls[str(pid)] = url

en_wiki = {}
for wid, url in conn.execute("SELECT wikidata_id, url FROM sitelinks WHERE site='en.wikipedia.org'"):
    en_wiki[wid] = url

iso_to_macro = {}
for iso, macro in conn.execute('SELECT DISTINCT iso_a3, macro_region FROM regions'):
    iso_to_macro[iso] = macro

print(f"Loaded {len(polity_urls):,} polity URLs, {len(en_wiki):,} en-Wikipedia URLs, {len(iso_to_macro)} country-to-macro mappings")

# Sample from each stratum
all_samples = []

for (method, origin), target_n in TARGETS.items():
    print(f"\n{'='*50}")
    print(f"Sampling {target_n} from method={method}, origin={origin}")

    cur = conn.execute('''
        SELECT ic.wikidata_id, ic.name_en, ic.polity_name, ic.polity_id,
               ic.method, ic.origin, ic.matched_name, ic.matched_wikidata_id,
               ic.impact_date,
               i.birthcity_en, i.deathcity_en, i.identifiers_count,
               ir.macro_region,
               ico.iso_a3_code
        FROM individuals_cliopatria ic
        JOIN individuals i ON ic.wikidata_id = i.wikidata_id
        LEFT JOIN individuals_regions ir ON ic.wikidata_id = ir.wikidata_id
        LEFT JOIN individuals_countries ico ON ic.wikidata_id = ico.wikidata_id
        WHERE ic.method = ? AND ic.origin = ?
    ''', (method, origin))

    pools = defaultdict(list)
    total = 0
    for row in cur:
        total += 1
        wid, name, polity, polity_id, meth, orig, matched, matched_wid, \
            impact, birth_city, death_city, id_count, macro, iso_a3 = row

        era = get_era(impact)
        notable = get_notability(id_count)

        if macro:
            macro_clean = macro.split(';')[0].strip()
        elif iso_a3:
            macro_clean = iso_to_macro.get(iso_a3, 'Unknown')
        else:
            macro_clean = 'Unknown'

        pools[(macro_clean, era, notable)].append(row)

    non_empty = {k: v for k, v in pools.items() if v}
    print(f"  Pool: {total:,} individuals across {len(non_empty)} sub-strata")

    if not non_empty:
        continue

    per_sub = max(1, target_n // len(non_empty))
    remaining = target_n
    selected = []
    selected_wids = set()

    for key in sorted(non_empty.keys()):
        pool = non_empty[key]
        n = min(per_sub, len(pool), remaining)
        picks = random.sample(pool, n)
        selected.extend(picks)
        selected_wids.update(r[0] for r in picks)
        remaining -= n
        if remaining <= 0:
            break

    if remaining > 0:
        leftover = []
        for pool in non_empty.values():
            for r in pool:
                if r[0] not in selected_wids:
                    leftover.append(r)
        if leftover:
            extra = random.sample(leftover, min(remaining, len(leftover)))
            selected.extend(extra)

    for row in selected:
        wid, name, polity, polity_id, meth, orig, matched, matched_wid, \
            impact, birth_city, death_city, id_count, macro, iso_a3 = row

        wiki_url = en_wiki.get(wid, f'https://www.wikidata.org/wiki/{wid}')

        city = ''
        if meth == 'polygon':
            if orig == 'birthplace':
                city = birth_city or ''
            elif orig == 'deathplace':
                city = death_city or ''
            elif orig == 'nationality':
                city = matched or ''

        matching_url = ''
        url_origin = ''
        if meth in ('url', 'url_fallback'):
            first_pid = polity_id.split(';')[0].strip() if polity_id else ''
            matching_url = polity_urls.get(first_pid, '')
            url_origin = f"{orig}: {matched or matched_wid or ''}"

        if macro:
            macro_clean = macro.split(';')[0].strip()
        elif iso_a3:
            macro_clean = iso_to_macro.get(iso_a3, 'Unknown')
        else:
            macro_clean = 'Unknown'

        era = get_era(impact)
        notable = get_notability(id_count)

        all_samples.append({
            'wikidata_id': wid,
            'name_en': name,
            'wikipedia_url': wiki_url,
            'polity_name': polity,
            'method': meth,
            'origin': orig,
            'matched_entity': matched or '',
            'city': city,
            'matching_url': matching_url,
            'url_origin': url_origin,
            'macro_region': macro_clean,
            'impact_date': impact if impact else '',
            'identifiers_count': id_count or 0,
            'strat_method': meth,
            'strat_origin': orig,
            'strat_macro_region': macro_clean,
            'strat_era': era,
            'strat_notability': notable,
        })

    regions = defaultdict(int)
    eras = defaultdict(int)
    for s in all_samples[-len(selected):]:
        regions[s['macro_region']] += 1
        eras[s['strat_era']] += 1
    print(f"  Selected: {len(selected)}")
    print(f"  Regions: {dict(sorted(regions.items(), key=lambda x: -x[1]))}")
    print(f"  Eras:    {dict(sorted(eras.items(), key=lambda x: -x[1]))}")

conn.close()
print(f"\n{'='*50}")
print(f"TOTAL SAMPLES: {len(all_samples)}")

Loaded 1,618 polity URLs, 2,114,047 en-Wikipedia URLs, 241 country-to-macro mappings

Sampling 120 from method=polygon, origin=nationality


  Pool: 3,918,101 individuals across 102 sub-strata


  Selected: 120
  Regions: {'Western Europe': 26, 'Asia': 18, 'Eastern Europe': 17, 'Middle-East and Africa (MENA)': 16, 'Sub-Saharan Africa': 15, 'Latin America': 9, 'North America': 9, 'Oceania': 7, 'Ancient Mediterranean': 3}
  Eras:    {'Contemporary': 38, 'Modern': 25, 'Early Modern': 23, 'Ancient': 19, 'Medieval': 15}

Sampling 60 from method=polygon, origin=birthplace


  Pool: 644,375 individuals across 115 sub-strata
  Selected: 60
  Regions: {'Asia': 15, 'Eastern Europe': 15, 'Latin America': 15, 'Middle-East and Africa (MENA)': 12, 'Ancient Mediterranean': 3}
  Eras:    {'Ancient': 15, 'Contemporary': 12, 'Early Modern': 12, 'Medieval': 12, 'Modern': 9}

Sampling 40 from method=polygon, origin=deathplace


  Pool: 101,549 individuals across 120 sub-strata
  Selected: 40
  Regions: {'Asia': 15, 'Eastern Europe': 15, 'Latin America': 7, 'Ancient Mediterranean': 3}
  Eras:    {'Ancient': 12, 'Contemporary': 9, 'Early Modern': 7, 'Medieval': 6, 'Modern': 6}

Sampling 30 from method=url, origin=nationality


...

  Selected: 120
  Regions: {'Eastern Europe': 14, 'Ancient Mediterranean': 13, 'Asia': 13, 'Latin America': 13, 'Western Europe': 13, 'Middle-East and Africa (MENA)': 12, 'North America': 12, 'Oceania': 12, 'Sub-Saharan Africa': 12, 'Unknown': 6}
  Eras:    {'Unknown': 120}

Sampling 55 from method=url_fallback, origin=birthplace


  Pool: 28,133 individuals across 29 sub-strata


  Selected: 55
  Regions: {'Eastern Europe': 9, 'North America': 9, 'Ancient Mediterranean': 7, 'Middle-East and Africa (MENA)': 7, 'Asia': 5, 'Oceania': 5, 'Western Europe': 5, 'Latin America': 3, 'Sub-Saharan Africa': 3, 'Unknown': 2}
  Eras:    {'Unknown': 55}

Sampling 40 from method=url_fallback, origin=deathplace


  Pool: 2,572 individuals across 23 sub-strata
  Selected: 40
  Regions: {'Latin America': 9, 'Ancient Mediterranean': 6, 'North America': 5, 'Asia': 4, 'Eastern Europe': 4, 'Middle-East and Africa (MENA)': 3, 'Sub-Saharan Africa': 3, 'Western Europe': 3, 'Oceania': 2, 'Unknown': 1}
  Eras:    {'Unknown': 40}



TOTAL SAMPLES: 500


In [ ]:
import pandas as pd

df_samples = pd.DataFrame(all_samples)
print(f"Validation sample: {len(df_samples)} individuals")
df_samples.head(20)

### Export to CSV and print summary statistics

The 500 sampled individuals are written to CSV with all relevant metadata and stratification labels.
A summary is printed showing the distribution across each stratification dimension.

In [3]:
os.makedirs('tables', exist_ok=True)

fieldnames = [
    'wikidata_id', 'name_en', 'wikipedia_url', 'polity_name',
    'method', 'origin', 'matched_entity', 'city', 'matching_url', 'url_origin',
    'macro_region', 'impact_date', 'identifiers_count',
    'strat_method', 'strat_origin', 'strat_macro_region', 'strat_era', 'strat_notability',
]

out_path = 'tables/validation_sample_500.csv'
with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_samples)

print(f"Written {len(all_samples)} rows to {out_path}")

# Summary statistics
print(f"\n{'='*50}")
print("SUMMARY")
print(f"{'='*50}")

for col in ['strat_method', 'strat_origin', 'strat_macro_region', 'strat_era', 'strat_notability']:
    counts = Counter(s[col] for s in all_samples)
    print(f"\n{col}:")
    for k, v in counts.most_common():
        print(f"  {k:30s}: {v:4d}")

n_enwiki = sum(1 for s in all_samples if 'en.wikipedia.org' in s['wikipedia_url'])
n_wikidata = sum(1 for s in all_samples if 'wikidata.org' in s['wikipedia_url'])
print(f"\nWikipedia URL coverage:")
print(f"  English Wikipedia: {n_enwiki}")
print(f"  Wikidata fallback: {n_wikidata}")

Written 500 rows to tables/validation_sample_500.csv

SUMMARY

strat_method:
  polygon                       :  220
  url_fallback                  :  215
  url                           :   65

strat_origin:
  nationality                   :  270
  birthplace                    :  135
  deathplace                    :   95

strat_macro_region:
  Eastern Europe                :  107
  Asia                          :   93
  Latin America                 :   58
  Middle-East and Africa (MENA) :   50
  Western Europe                :   47
  Ancient Mediterranean         :   42
  North America                 :   35
  Sub-Saharan Africa            :   33
  Oceania                       :   26
  Unknown                       :    9

strat_era:
  Unknown                       :  215
  Contemporary                  :   78
  Ancient                       :   56
  Modern                        :   56
  Early Modern                  :   51
  Medieval                      :   44

strat_notability